In [1]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
build_and_test.sh   LICENSE	    README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	    SECURITY.md  tests
experiments	    pyproject.toml  src		 test.sh


In [2]:
!pip install -e .

Obtaining file:///content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.5/510.5 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 147.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 132.7 MB/s eta 0:00:00
  Building editable for transformercompression (pyproject.toml) ... done
  Created wheel for trans

In [2]:
import slicegpt
from slicegpt import rotate, model_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)

slicegpt imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/.shortcut-targets-by-id/1VRkguTVtRkllTlSID9ugy4MHwCacsE_K/Pratikum/code/sliced_rag/SliceGPTModifications/src/slicegpt/model_utils.py


In [3]:
import os, textwrap

# Where to save logs and models in your Drive
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
LOG_DIR = os.path.join(BASE_RESULTS_DIR, "logs_opt_pca/Qwen2-0.5B-coqa")
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_opt_pca/Qwen2-0.5B-coqa")


os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [6]:
import subprocess
from datetime import datetime

def run_slicegpt(dataset, sparsity):
    log_name = f"{dataset}_s{sparsity:.2f}".replace(".", "p") + ".txt" # 0.25 -> 0p25
    log_path = os.path.join(LOG_DIR, log_name)

    # if os.path.exists(log_path):
    #     print(f"[SKIP] Log already exists for {dataset}, sparsity={sparsity}: {log_path}")
    #     return

    # save_dir = os.path.join(MODEL_DIR, f"{dataset}_s{sparsity:.2f}".replace(".", "p"))
    # os.makedirs(save_dir, exist_ok=True)

    save_dir = "/content/drive/MyDrive/TUM/temp/qwen_coqa/"
    os.makedirs(save_dir, exist_ok=True)

    cmd = [
        "python",
        "/content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/experiments/run_slicegpt.py",
        "--model", "Qwen/Qwen2-0.5B",
        "--cal-dataset", dataset,
        "--save-dir", save_dir,
        "--sparsity", str(sparsity),
        "--device", "cuda:0",
        "--no-wandb",
        "--cal-batch-size", "8",
        "--save-hidden-states",
        "--save-rotation-matrices"
    ]

    print("\n=====================================================")
    print("Running:", " ".join(cmd))
    print("Log file:", log_path)
    print("Start:", datetime.now())
    print("=====================================================\n")

    with open(log_path, "w") as f:
        # Stream output to both cell and file
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            print(line, end="")   # to notebook
            f.write(line)         # to log file

    ret = process.wait()
    print("\nFinished with return code:", ret)
    print("End:", datetime.now())


In [8]:
#we do the first run just getting the rotation matrices with sparsity = 0.0
datasets = ["coqa"]
sparsities = [0.0]

for dataset in datasets:
    for s in sparsities:
        run_slicegpt(dataset, s)


Running: python /content/drive/MyDrive/TUM/Pratikum/code/sliced_rag/SliceGPTModifications/experiments/run_slicegpt.py --model Qwen/Qwen2-0.5B --cal-dataset coqa --save-dir /content/drive/MyDrive/TUM/temp/qwen_coqa/ --sparsity 0.0 --device cuda:0 --no-wandb --cal-batch-size 8 --save-hidden-states --save-rotation-matrices
Log file: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/logs_opt_pca/Qwen2-0.5B-coqa/coqa_s0p00.txt
Start: 2026-02-09 19:40:33.084205

Running SliceGPT experiment.
PyTorch device: cuda:0
Number of available cuda devices: 1
Loading Qwen/Qwen2-0.5B config and model weights from Hugging Face
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading model done
Loading dataset: coqa
Loading dataset done
Preparing dataloader
Preparing dataloader done
Preparing test dataloader
Token indices sequence length is longer than the specified maximum sequence length for this model (114213 > 32768). Running 